## Method 1
#### Agent help to suggest the filename with the help of prompt and also the link.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_groq.chat_models import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",    
    # reasoning_effort= "parsed"  # disable thinking
    reasoning_format="hidden",
)

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/Users/vansh/Projects/arduchat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25646.38it/s]


In [2]:
from langchain_community.document_loaders import JSONLoader

json_data = JSONLoader(
    "../web-crawller/links_result_deduped.json",
    jq_schema=".[]",
    text_content=True
).load()
json_data

/var/folders/14/mdg60hc171q8ft30dvw15r7m0000gn/T/ipykernel_52546/3560564827.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader


[Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 1}, page_content='https://ardupilot.org/copter/docs'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 2}, page_content='https://ardupilot.org/copter/docs/ArduCopter_MAVLink_Messages.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 3}, page_content='https://ardupilot.org/copter/docs/ac2_followme.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 4}, page_content='https://ardupilot.org/copter/docs/ac2_guidedmode.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 5}, page_content='https://ardupilot.org/copter/docs/ac2_positionmode.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/

In [3]:
json_data[930]

Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 931}, page_content='https://ardupilot.org/copter/docs/www.DJI.com')

In [4]:
from langchain_community.vectorstores import Chroma

links_vector_db = Chroma.from_documents(
    documents=json_data,
    embedding=embedding
)

In [5]:
link_retriever = links_vector_db.as_retriever()
link_retriever.invoke("parameters")

[Document(metadata={'seq_num': 827, 'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/parameters.html'),
 Document(metadata={'seq_num': 569, 'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json'}, page_content='https://ardupilot.org/copter/docs/common-scripting-parameters.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 897}, page_content='https://ardupilot.org/copter/docs/traditional-helicopter-parameter-list.html'),
 Document(metadata={'source': '/Users/vansh/Projects/arduchat/web-crawller/links_result_deduped.json', 'seq_num': 774}, page_content='https://ardupilot.org/copter/docs/parameters-Copter-beta-V4.6.3.html')]

In [6]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. 

Here is the junior question : {question},

now your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions
Just give a array, donot give any thing else
""")

prompt.invoke({"question": "what is circle mode"})

StringPromptValue(text='\nYou are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. \n\nHere is the junior question : what is circle mode,\n\nnow your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions\nJust give a array, donot give any thing else\n')

In [7]:
prompt.invoke({"question":"what is circle mode"})

StringPromptValue(text='\nYou are a senior drone engineer and you are mentoring the juniors on the Ardupilot Copter documentation. \n\nHere is the junior question : what is circle mode,\n\nnow your task is to give set of keywords so that retriever can easily find the links that may be necessary for the answering the questions\nJust give a array, donot give any thing else\n')

In [8]:
chain_keyword = prompt | llm

In [9]:
link_keyword = chain_keyword.invoke({"question":"what is circle mode"}).content

In [10]:
link_docs = link_retriever.invoke(link_keyword)
retrieved_urls = [doc.page_content for doc in link_docs]
retrieved_urls

['https://ardupilot.org/copter/docs/flight-modes.html',
 'https://ardupilot.org/copter/docs/circle-mode.html',
 'https://ardupilot.org/copter/docs/copter-flight-features.html',
 'https://ardupilot.org/copter/docs/auto-mode.html']

In [11]:
already_fetched_urls = set()

url_to_be_searched = []

def add_retrieved_urls(new_urls):
    global url_to_be_searched
    url_to_be_searched = [u for u in new_urls if u not in already_fetched_urls]

def mark_as_fetched():
    already_fetched_urls.update(url_to_be_searched)

In [12]:
add_retrieved_urls(retrieved_urls)
mark_as_fetched()

In [13]:
from langchain_community.document_loaders import WebBaseLoader

docs = WebBaseLoader(url_to_be_searched).load()
docs

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://ardupilot.org/copter/docs/flight-modes.html', 'title': 'Flight Modes — Copter  documentation', 'language': 'en'}, page_content="\n\n\n\n\nFlight Modes — Copter  documentation\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nHome\n\nCopter\nPlane\nRover\nBlimp\nSub\nAntennaTracker\nMission Planner\nAPM Planner 2\nMAVProxy\nCompanion Computers\nDeveloper\n\n\nDownloads\n\nMission Planner\nAPM Planner 2\nAdvanced User Tools\nDeveloper Tools\nFirmware\n\n\nCommunity\n\nSupport Forums\nFacebook\nDeveloper Chat (Discord)\nDeveloper Voice (Discord)\nContact us\nGetting involved\nCommercial Support\nDevelopment Team\nUAS Training Centers\n\n\nStores\nAbout\n\nNews\nHistory\nLicense\nTrademark\nAcknowledgments\nWiki Editing Guide\nPartners Program\n\n\n\n\n\n\n\n\n\n            Copter\n              \n\n\n\n\n\n\n\n\n\n\nIntroduction to Copter\nChoosing an Autopilot\nGround Control Stations\nFirst Time Setup\nInstall Ground Station Software\nAutopilot System A

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splitted_docs = splitter.split_documents(docs)

In [15]:
vectordb = Chroma.from_documents(
    documents = splitted_docs,
    embedding = embedding
)
vectordb

In [16]:
main_retriever = vectordb.as_retriever()

In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt2 = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using the context below.\n\nContext:\n{context}. In case, if you provide a link make sure it is a working link and from either {already_visited} or {retrieved_links}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chain = (
    {
        "context": lambda x: main_retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "history": lambda x: x["history"],   # <-- pass it through
        "already_visited": lambda x: list(already_fetched_urls),
        "retrieved_links": lambda x: list(url_to_be_searched),
    }
    | prompt2
    | llm
)

# store: session_id -> history object
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

In [24]:
from IPython.display import Markdown
config = {"configurable": {"session_id": "user_1"}}

r1 = chain_with_history.invoke({"question": "what is circle mode"}, config=config)
Markdown(r1.content)

r2 = chain_with_history.invoke({"question": "what's its radius limit?"}, config=config)
Markdown(r2.content)

The radius limit for **Circle Mode** in Ardupilot is determined by the `CIRCLE_RADIUS_M` parameter. While the exact allowable range isn't explicitly mentioned in the provided context, typical firmware configurations allow a radius range of **0 to 500 meters** (this can vary slightly depending on the Ardupilot version). 

- **Minimum Radius**: 0 meters (though practical use requires a positive value to maintain a circular path).
- **Maximum Radius**: Often capped at 500 meters to prevent excessively large orbits.

For precise values and updates, refer to the official [Circle Mode documentation](https://ardupilot.org/copter/docs/circle-mode.html) or check the parameter description in your Ardupilot configuration tool (e.g., Mission Planner or QGroundControl).

In [25]:
r2 = chain_with_history.invoke({"question": "what's its radius limit?"}, config=config)
Markdown(r2.content)

The **radius limit** for **Circle Mode** in Ardupilot is defined by the `CIRCLE_RADIUS_M` parameter, which is specified in **meters**. While the official documentation does not explicitly state a strict maximum radius limit, practical firmware implementations (e.g., Ardupilot) typically allow a configurable range of **0 to 500 meters** for the radius. 

- **Minimum**: 0 meters (though a radius of 0 effectively disables circular motion).
- **Maximum**: Often capped at **500 meters** (but this can vary slightly depending on the Ardupilot version or firmware update).

For precise details, refer to the [Circle Mode documentation](https://ardupilot.org/copter/docs/circle-mode.html) or check the parameter description in your configuration tool (e.g., Mission Planner or QGroundControl). The exact limit is determined by the firmware's parameter constraints.

In [26]:
r2 = chain_with_history.invoke({"question": "what is follow me mode?"}, config=config)
Markdown(r2.content)

**Follow Me Mode** is a flight mode in Ardupilot (for Copter) where the drone autonomously follows a moving target, such as a person, vehicle, or ground control station, maintaining a predefined distance and altitude relative to the target. This mode uses GPS or other position data to track the target's movement and adjust the drone's position accordingly.

### Key Features:
- **Relative Positioning**: The drone maintains a fixed horizontal distance (e.g., 5 meters to the side) and altitude (e.g., 10 meters above) from the target.
- **Smooth Tracking**: The drone adjusts its flight path in real time to follow the target's movement, even during turns or changes in speed.
- **Activation**: Typically activated via a ground control station (e.g., Mission Planner) or a dedicated remote control command.

### Parameters (Customizable):
- `FOLLOW_RADIUS`: Sets the horizontal distance to maintain from the target.
- `FOLLOW_ALT`: Sets the altitude (above ground or above target) to maintain.
- `FOLLOW_TYPE`: Determines the relative position (e.g., behind, beside, or above the target).
- `FOLLOW_YAW`: Controls whether the drone faces the target or follows a fixed direction.

### Limitations:
- Relies on a stable GPS signal for accurate tracking.
- May not work optimally in environments with GPS interference or poor signal.
- The target must be within GPS range (typically 10 km or less, depending on firmware).

For detailed configuration and parameter explanations, refer to the **[Follow Mode documentation](https://ardupilot.org/copter/docs/follow-mode.html)**.

In [27]:
r2 = chain_with_history.invoke({"question": "whats the difference between circle mode and follow me mode?"}, config=config)
Markdown(r2.content)

The **Circle Mode** and **Follow Me Mode** in Ardupilot serve distinct purposes and operate differently. Here's a breakdown of their key differences:

---

### **1. Target Behavior**  
- **Circle Mode**:  
  - The drone orbits a **fixed point** (e.g., a location or a stationary object) in a circular path.  
  - The center of the circle is defined by the `CIRCLE_RADIUS_M` parameter (in meters).  
  - The drone maintains a constant radius and angular velocity (adjustable via control sticks if enabled).  

- **Follow Me Mode**:  
  - The drone autonomously tracks a **moving target** (e.g., a person, vehicle, or ground station) using GPS data.  
  - The drone maintains a **relative position** (distance and altitude) from the target (e.g., 10 meters behind and 5 meters above).  

---

### **2. Movement Type**  
- **Circle Mode**:  
  - The drone follows a **fixed-radius circular path** around a point.  
  - The pilot can manually adjust the radius and angular velocity using control sticks (if `CIRCLE_OPTIONS` is enabled).  

- **Follow Me Mode**:  
  - The drone **dynamically adjusts** its position to follow the target's movement in real time.  
  - The drone’s flight path is not pre-defined; it reacts to the target’s movement (e.g., turns, speed changes).  

---

### **3. Control Input**  
- **Circle Mode**:  
  - The pilot can adjust the **circle’s radius and angular velocity** using control sticks (if `CIRCLE_OPTIONS` is enabled).  

- **Follow Me Mode**:  
  - The drone operates **autonomously** based on GPS data and predefined parameters (e.g., `FOLLOW_RADIUS`, `FOLLOW_ALT`).  
  - The pilot cannot manually override the drone’s behavior during active tracking (except to disengage the mode).  

---

### **4. Use Cases**  
- **Circle Mode**:  
  - Ideal for **surveillance, mapping, or filming a stationary object** (e.g., a landmark) from a circular orbit.  

- **Follow Me Mode**:  
  - Useful for **filming a moving subject** (e.g., a hiker, car, or athlete) while maintaining a consistent relative position.  

---

### **5. Key Parameters**  
- **Circle Mode**:  
  - `CIRCLE_RADIUS_M` (radius of the circle, in meters).  
  - `CIRCLE_OPTIONS` (controls stick-based adjustments).  

- **Follow Me Mode**:  
  - `FOLLOW_RADIUS` (horizontal distance from the target).  
  - `FOLLOW_ALT` (altitude above the target).  
  - `FOLLOW_TYPE` (relative position: behind, beside, or above the target).  

---

### **6. Limitations**  
- **Circle Mode**:  
  - Requires manual piloting (via sticks) to adjust orbit parameters if enabled.  
  - Limited to orbiting a fixed point.  

- **Follow Me Mode**:  
  - Relies on **GPS accuracy** for tracking.  
  - May struggle in environments with GPS signal loss or interference.  

---

### **Summary Table**  

| Feature                | **Circle Mode**                          | **Follow Me Mode**                          |
|------------------------|------------------------------------------|---------------------------------------------|
| **Target**             | Fixed point                              | Moving target (e.g., person/vehicle)          |
| **Path**               | Circular orbit                           | Dynamic position relative to target           |
| **Control Input**      | Adjust radius/angular velocity via sticks (if enabled) | Fully autonomous tracking                   |
| **Use Case**           | Surveillance/mapping stationary objects  | Filming/monitoring moving subjects            |
| **Parameters**         | `CIRCLE_RADIUS_M`, `CIRCLE_OPTIONS`        | `FOLLOW_RADIUS`, `FOLLOW_ALT`, `FOLLOW_TYPE` |

---

For more details:  
- [Circle Mode Documentation](https://ardupilot.org/copter/docs/circle-mode.html)  
- [Follow Me Mode Documentation](https://ardupilot.org/copter/docs/follow-mode.html)

In [28]:
r2 = chain_with_history.invoke({"question": "what are the parameters that are related to motor movements?"}, config=config)
Markdown(r2.content)

Here are the key **parameters related to motor movements** in Ardupilot (Copter), based on the context provided:

---

### **1. Motor Control Parameters**
- **`MOT_PWM_Range`**: Defines the PWM range used to control motor speed. Typically set to **1100 to 2000 µs** for most ESCs.  
- **`MOT_PWM_MIN`/`MOT_PWM_MAX`**: Sets the minimum and maximum PWM values sent to motors.  
- **`MOT_THR_MIN`/`MOT_THR_MAX`**: Maps throttle input (from 0 to 100%) to motor output (e.g., 15% to 95% thrust).  
- **`MOT_SPIN_ARMED`**: Ensures motors spin only when armed (prevents accidental motor startup).  

---

### **2. Motor Mixing Parameters**  
- **`MIXER_0` to `MIXER_3`**: Define the motor mixing ratios for multirotor configurations (e.g., quad, hex, octo). These map stick inputs to motor outputs.  
- **`MIXER_TYPE`**: Specifies the type of mixer (e.g., quad, Y6, etc.).  

---

### **3. Motor Arming Parameters**  
- **`ARMING_CHECK`**: Enables/disables checks before arming (e.g., GPS, battery status).  
- **`ARMING_REQUIRE`**: Sets the condition for arming (e.g., throttle at minimum, safety switch engaged).  

---

### **4. Motor Thrust Scaling**  
- **`MOT_THRUST_SCALE`**: Adjusts how motor thrust increases with throttle input (critical for accurate flight dynamics).  
- **`MOT_THRUST_MIN`/`MOT_THRUST_MAX`**: Sets the minimum and maximum thrust output values.  

---

### **5. Motor Fail-Safe Parameters**  
- **`FS_THR_ENABLE`**: Enables throttle-based motor failsafe (e.g., if throttle drops below a threshold, motors stop).  
- **`FS_THR_VALUE`**: Sets the throttle value that triggers a failsafe.  

---

### **6. Motor Range and Calibration**  
- **`MOT_RANGE`**: Sets the maximum range (in meters) for motor output scaling (used in advanced configurations).  
- **`MOT_CAL`**: Enables calibration of motor outputs for precise thrust control.  

---

### **7. Motor Start/Stop Behavior**  
- **`MOT_START_DELAY`**: Delays motor startup after arming (useful for safety).  
- **`MOT_STOP_DELAY`**: Delays motor shutdown after disarming.  

---

### **Key Documentation References**  
For detailed explanations, refer to the following sections of the Ardupilot documentation:  
- [Generating Motor Diagrams](https://ardupilot.org/copter/docs/common-generating-motor-diagrams.html)  
- [Motor Thrust Scaling](https://ardupilot.org/copter/docs/motor-thrust-scaling.html)  
- [Arming the Motors](https://ardupilot.org/copter/docs/arming_the_motors.html)  
- [Set Motor Range](https://ardupilot.org/copter/docs/set-motor-range.html)  

> **Note**: Always adjust these parameters using a ground control station (e.g., Mission Planner or QGroundControl) to ensure safe and stable motor behavior.